# Tutorial 04: VRP Problem Variants

Learn about different Vehicle Routing Problem (VRP) variants and how to model them using vrp-toolkit.

**What you'll learn:**
- Understand different VRP problem types (VRP, CVRP, PDP, PDPTW)
- Model each variant using PDPTWInstance
- Know when to use each problem type
- Convert between problem variants

**Prerequisites:**
- Tutorial 01 (Quickstart)
- Tutorial 03 (Custom Problems)

**Time:** ~35 minutes

## 1. Setup and Imports

In [ ]:
# Standard imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# VRP Toolkit imports
from vrp_toolkit.problems.pdptw import PDPTWInstance, Node
from vrp_toolkit.algorithms.alns.solver import ALNSSolver

# Verify imports
print("All imports successful!")

## 2. Quick Start: VRP Hierarchy

Let's understand the VRP problem hierarchy with a simple example.

**VRP Family:**
```
VRP (Vehicle Routing Problem)
└── CVRP (Capacitated VRP) - adds vehicle capacity
    └── VRPTW (VRP with Time Windows) - adds time constraints
        └── PDP (Pickup and Delivery) - adds paired pickups/deliveries
            └── PDPTW (PDP with Time Windows) - most complex
```

In [ ]:
# Simple VRP: Visit 3 customers from depot
# (We model this as PDPTW with relaxed constraints)

vrp_nodes = [
    Node(node_id=0, x=0, y=0, node_type='depot'),
    # Customer 1 (modeled as pickup with paired dummy delivery at depot)
    Node(node_id=1, x=5, y=5, demand=1.0, time_window=(0, 1000), service_time=1, node_type='pickup', pair_node_id=2),
    Node(node_id=2, x=0, y=0, demand=-1.0, time_window=(0, 1000), service_time=0, node_type='delivery', pair_node_id=1),
    # Customer 2
    Node(node_id=3, x=10, y=3, demand=1.0, time_window=(0, 1000), service_time=1, node_type='pickup', pair_node_id=4),
    Node(node_id=4, x=0, y=0, demand=-1.0, time_window=(0, 1000), service_time=0, node_type='delivery', pair_node_id=3),
    # Customer 3
    Node(node_id=5, x=8, y=8, demand=1.0, time_window=(0, 1000), service_time=1, node_type='pickup', pair_node_id=6),
    Node(node_id=6, x=0, y=0, demand=-1.0, time_window=(0, 1000), service_time=0, node_type='delivery', pair_node_id=5),
]

simple_vrp = PDPTWInstance(
    nodes=vrp_nodes,
    battery_capacity=1000.0,  # Relaxed
    max_route_time=1000.0,    # Relaxed
    vehicle_capacity=10.0
)

print(f"Created simple VRP with {simple_vrp.n} customers")

**What just happened:**
- VRP has only depot + customers (no explicit pickups/deliveries)
- We modeled customers as "pickup" nodes with dummy deliveries at depot
- This trick allows using PDPTWInstance for simpler VRP variants
- Relaxed constraints (large time windows, capacity) make it a basic VRP

## 3. Understanding VRP Variants

### 3.1 VRP (Basic Vehicle Routing Problem)

**Problem:** Visit a set of customers from a depot, minimize total distance.

**Constraints:**
- Start and end at depot
- Visit each customer exactly once

**NO constraints on:**
- Vehicle capacity
- Time windows
- Pickup-delivery pairing

In [ ]:
def create_vrp(customer_locations, num_vehicles=1):
    """
    Create a basic VRP instance.
    
    Args:
        customer_locations: List of (x, y) tuples
        num_vehicles: Number of vehicles available
    """
    nodes = [Node(node_id=0, x=0, y=0, node_type='depot')]
    
    # For each customer, create pickup + dummy delivery at depot
    for i, (x, y) in enumerate(customer_locations, 1):
        pickup_id = 2 * i - 1
        delivery_id = 2 * i
        
        nodes.append(Node(
            node_id=pickup_id,
            x=x, y=y,
            demand=1.0,
            time_window=(0, 10000),  # Very relaxed
            service_time=1.0,
            node_type='pickup',
            pair_node_id=delivery_id
        ))
        
        # Dummy delivery at depot (cost = 0)
        nodes.append(Node(
            node_id=delivery_id,
            x=0, y=0,
            demand=-1.0,
            time_window=(0, 10000),
            service_time=0,
            node_type='delivery',
            pair_node_id=pickup_id
        ))
    
    return PDPTWInstance(
        nodes=nodes,
        battery_capacity=100000.0,  # Effectively unlimited
        max_route_time=100000.0,
        vehicle_capacity=100000.0
    )

# Example: 5 customer VRP
customers = [(10, 5), (15, 10), (8, 12), (20, 8), (12, 15)]
vrp_instance = create_vrp(customers)

print(f"VRP instance: {len(customers)} customers")
solver = ALNSSolver()
vrp_solution = solver.solve(vrp_instance)
print(f"Total distance: {vrp_solution.objective_value():.2f}")

### 3.2 CVRP (Capacitated VRP)

**Problem:** VRP + vehicle capacity constraints.

**Additional constraints:**
- Each customer has a demand
- Vehicle has limited capacity
- Total demand on route ≤ vehicle capacity

**Use case:** Delivery trucks with weight/volume limits

In [ ]:
def create_cvrp(customer_locations, customer_demands, vehicle_capacity):
    """
    Create a CVRP instance.
    
    Args:
        customer_locations: List of (x, y) tuples
        customer_demands: List of demand values (positive)
        vehicle_capacity: Maximum vehicle capacity
    """
    nodes = [Node(node_id=0, x=0, y=0, node_type='depot')]
    
    for i, ((x, y), demand) in enumerate(zip(customer_locations, customer_demands), 1):
        pickup_id = 2 * i - 1
        delivery_id = 2 * i
        
        # Customer location (pickup)
        nodes.append(Node(
            node_id=pickup_id,
            x=x, y=y,
            demand=demand,  # Actual demand
            time_window=(0, 10000),
            service_time=1.0,
            node_type='pickup',
            pair_node_id=delivery_id
        ))
        
        # Dummy delivery at depot
        nodes.append(Node(
            node_id=delivery_id,
            x=0, y=0,
            demand=-demand,
            time_window=(0, 10000),
            service_time=0,
            node_type='delivery',
            pair_node_id=pickup_id
        ))
    
    return PDPTWInstance(
        nodes=nodes,
        battery_capacity=100000.0,
        max_route_time=100000.0,
        vehicle_capacity=vehicle_capacity  # KEY: Limited capacity
    )

# Example: CVRP with capacity constraint
customers = [(10, 5), (15, 10), (8, 12), (20, 8), (12, 15)]
demands = [5, 8, 3, 7, 4]  # Total = 27
capacity = 15  # Need at least 2 vehicles!

cvrp_instance = create_cvrp(customers, demands, capacity)

print(f"CVRP instance: {len(customers)} customers")
print(f"Total demand: {sum(demands)} | Vehicle capacity: {capacity}")
print(f"Minimum vehicles needed: {int(np.ceil(sum(demands) / capacity))}")

cvrp_solution = solver.solve(cvrp_instance)
print(f"\nSolution: {len(cvrp_solution.routes)} routes")
print(f"Total distance: {cvrp_solution.objective_value():.2f}")

### 3.3 VRPTW (VRP with Time Windows)

**Problem:** VRP + time window constraints.

**Additional constraints:**
- Each customer has time window [earliest, latest]
- Must arrive within time window
- Can wait if arrive early

**Use case:** Deliveries with customer availability ("deliver between 9am-12pm")

In [ ]:
def create_vrptw(customer_locations, time_windows):
    """
    Create a VRPTW instance.
    
    Args:
        customer_locations: List of (x, y) tuples
        time_windows: List of (earliest, latest) tuples
    """
    nodes = [Node(node_id=0, x=0, y=0, node_type='depot')]
    
    for i, ((x, y), tw) in enumerate(zip(customer_locations, time_windows), 1):
        pickup_id = 2 * i - 1
        delivery_id = 2 * i
        
        # Customer with time window
        nodes.append(Node(
            node_id=pickup_id,
            x=x, y=y,
            demand=1.0,
            time_window=tw,  # KEY: Time window constraint
            service_time=1.0,
            node_type='pickup',
            pair_node_id=delivery_id
        ))
        
        # Dummy delivery
        nodes.append(Node(
            node_id=delivery_id,
            x=0, y=0,
            demand=-1.0,
            time_window=(0, 10000),
            service_time=0,
            node_type='delivery',
            pair_node_id=pickup_id
        ))
    
    return PDPTWInstance(
        nodes=nodes,
        battery_capacity=100000.0,
        max_route_time=100000.0,
        vehicle_capacity=100000.0
    )

# Example: VRPTW with morning and afternoon time windows
customers = [(10, 5), (15, 10), (8, 12)]
time_windows = [
    (8, 12),   # Morning customer
    (13, 17),  # Afternoon customer
    (8, 17)    # All-day customer
]

vrptw_instance = create_vrptw(customers, time_windows)

print("VRPTW instance with time windows:")
for i, tw in enumerate(time_windows, 1):
    print(f"  Customer {i}: {tw}")

vrptw_solution = solver.solve(vrptw_instance)
print(f"\nTotal distance: {vrptw_solution.objective_value():.2f}")

### 3.4 PDP (Pickup and Delivery Problem)

**Problem:** Transport items from pickup locations to delivery locations.

**Key difference from VRP:**
- Pickup and delivery are **different locations**
- Each request has paired pickup + delivery
- Must visit pickup **before** delivery

**Use case:** Taxi services, package delivery, moving services

In [ ]:
# Example: Simple PDP (no time windows)
pdp_nodes = [
    Node(node_id=0, x=0, y=0, node_type='depot'),
    
    # Request 1: Pickup at (5,5), deliver to (15,10)
    Node(node_id=1, x=5, y=5, demand=2.0, time_window=(0, 1000), service_time=1, node_type='pickup', pair_node_id=2),
    Node(node_id=2, x=15, y=10, demand=-2.0, time_window=(0, 1000), service_time=1, node_type='delivery', pair_node_id=1),
    
    # Request 2: Pickup at (8,3), deliver to (12,12)
    Node(node_id=3, x=8, y=3, demand=3.0, time_window=(0, 1000), service_time=1, node_type='pickup', pair_node_id=4),
    Node(node_id=4, x=12, y=12, demand=-3.0, time_window=(0, 1000), service_time=1, node_type='delivery', pair_node_id=3),
]

pdp_instance = PDPTWInstance(
    nodes=pdp_nodes,
    battery_capacity=100000.0,
    max_route_time=100000.0,
    vehicle_capacity=10.0
)

print("PDP instance:")
print(f"  Request 1: ({pdp_nodes[1].x}, {pdp_nodes[1].y}) → ({pdp_nodes[2].x}, {pdp_nodes[2].y})")
print(f"  Request 2: ({pdp_nodes[3].x}, {pdp_nodes[3].y}) → ({pdp_nodes[4].x}, {pdp_nodes[4].y})")

pdp_solution = solver.solve(pdp_instance)
print(f"\nSolution routes: {pdp_solution.routes}")
print(f"Total distance: {pdp_solution.objective_value():.2f}")

### 3.5 PDPTW (Pickup and Delivery with Time Windows)

**Problem:** PDP + time window constraints (most complex variant).

**All constraints:**
- Pickup before delivery (PDP)
- Vehicle capacity (CVRP)
- Time windows (VRPTW)
- Battery constraints (optional)

**Use case:** On-demand delivery, ride-sharing, same-day shipping

In [ ]:
# Example: Full PDPTW with all constraints
pdptw_nodes = [
    Node(node_id=0, x=0, y=0, node_type='depot'),
    
    # Request 1: Morning pickup, afternoon delivery
    Node(node_id=1, x=10, y=5, demand=3.0, time_window=(8, 12), service_time=0.5, node_type='pickup', pair_node_id=2),
    Node(node_id=2, x=20, y=10, demand=-3.0, time_window=(13, 17), service_time=0.5, node_type='delivery', pair_node_id=1),
    
    # Request 2: All-day availability, but must be quick
    Node(node_id=3, x=5, y=15, demand=2.0, time_window=(9, 16), service_time=0.5, node_type='pickup', pair_node_id=4),
    Node(node_id=4, x=15, y=20, demand=-2.0, time_window=(10, 17), service_time=0.5, node_type='delivery', pair_node_id=3),
]

pdptw_instance = PDPTWInstance(
    nodes=pdptw_nodes,
    battery_capacity=100.0,     # Battery constraint
    max_route_time=480.0,       # 8-hour shift
    vehicle_capacity=5.0        # Limited capacity
)

print("PDPTW instance (full constraints):")
print(f"  Vehicle capacity: {pdptw_instance.vehicle_capacity}")
print(f"  Battery capacity: {pdptw_instance.battery_capacity}")
print(f"  Max route time: {pdptw_instance.max_route_time}")

pdptw_solution = solver.solve(pdptw_instance)
print(f"\nSolution:")
print(f"  Routes: {pdptw_solution.routes}")
print(f"  Total distance: {pdptw_solution.objective_value():.2f}")
print(f"  Feasible: {pdptw_solution.is_feasible()}")

## 4. Advanced: Problem Conversion

Sometimes you need to convert between problem types.

### 4.1 VRP → CVRP

**How:** Add customer demands and set vehicle capacity

In [ ]:
# Start with VRP
vrp_base = create_vrp([(10, 5), (15, 10), (8, 12)])

# Convert to CVRP by changing capacity and demands
vrp_base.vehicle_capacity = 6.0  # Add capacity constraint

# Update demands (currently all 1.0)
for node in vrp_base.nodes:
    if node.node_type == 'pickup':
        node.demand = 3.0  # Increase demand
    elif node.node_type == 'delivery':
        node.demand = -3.0

print("Converted VRP to CVRP")
print(f"New capacity: {vrp_base.vehicle_capacity}")
print(f"Customer demands: {[n.demand for n in vrp_base.nodes if n.node_type == 'pickup']}")

### 4.2 CVRP → VRPTW

**How:** Add time windows to nodes

In [ ]:
# Start with CVRP
cvrp_base = create_cvrp([(10, 5), (15, 10)], [4, 3], 10)

# Add time windows to convert to CVRPTW
time_windows_list = [(8, 12), (13, 17)]  # Morning and afternoon

pickup_idx = 0
for node in cvrp_base.nodes:
    if node.node_type == 'pickup':
        node.time_window = time_windows_list[pickup_idx]
        pickup_idx += 1

print("Converted CVRP to CVRPTW")
for node in cvrp_base.nodes:
    if node.node_type == 'pickup':
        print(f"  Node {node.node_id}: time window {node.time_window}")

## 5. Real-World Example: Multi-Variant Comparison

Let's solve the same scenario as different problem types and compare results.

In [ ]:
# Common scenario: 6 customer locations
locations = [
    (10, 5), (15, 10), (8, 12),
    (20, 8), (12, 15), (5, 18)
]

# Create 3 variants
print("Creating problem variants...\n")

# 1. VRP (no constraints)
vrp_comparison = create_vrp(locations)
sol_vrp = solver.solve(vrp_comparison)

# 2. CVRP (capacity = 10, demands = [3,4,2,5,3,4])
demands = [3, 4, 2, 5, 3, 4]
cvrp_comparison = create_cvrp(locations, demands, vehicle_capacity=10)
sol_cvrp = solver.solve(cvrp_comparison)

# 3. VRPTW (add time windows)
tws = [(8, 12), (9, 13), (10, 14), (13, 17), (14, 18), (15, 19)]
vrptw_comparison = create_vrptw(locations, tws)
sol_vrptw = solver.solve(vrptw_comparison)

# Compare results
print("Comparison Results:")
print("=" * 60)
print(f"{'Problem':<15} {'Routes':<10} {'Distance':<15} {'Feasible'}")
print("-" * 60)
print(f"{'VRP':<15} {len(sol_vrp.routes):<10} {sol_vrp.objective_value():<15.2f} {sol_vrp.is_feasible()}")
print(f"{'CVRP':<15} {len(sol_cvrp.routes):<10} {sol_cvrp.objective_value():<15.2f} {sol_cvrp.is_feasible()}")
print(f"{'VRPTW':<15} {len(sol_vrptw.routes):<10} {sol_vrptw.objective_value():<15.2f} {sol_vrptw.is_feasible()}")
print("=" * 60)

print("\nObservations:")
print("- VRP: Usually shortest distance (fewest constraints)")
print("- CVRP: May need more routes due to capacity")
print("- VRPTW: May have longer routes due to time windows")

## 6. When to Use Each Variant

**Use VRP when:**
- Simple routing without capacity or time constraints
- Initial planning or rough estimates
- Teaching basic routing concepts

**Use CVRP when:**
- Vehicle capacity is limiting factor
- Delivery trucks with weight/volume constraints
- No time constraints

**Use VRPTW when:**
- Customers have availability windows
- Service appointments
- No pickup-delivery pairing

**Use PDP when:**
- Items must go from pickup to delivery
- Taxi/rideshare services
- No time constraints

**Use PDPTW when:**
- Full real-world scenario
- Pickup-delivery with time windows
- Same-day delivery, moving services
- Electric vehicles (battery constraints)

**Problem Complexity Ranking:**
```
VRP < CVRP ≈ VRPTW < PDP < PDPTW
(easiest)                    (hardest)
```

## 7. Practice Exercises

1. **Basic:** Create a CVRP instance with 4 customers where total demand exceeds vehicle capacity. Verify it uses multiple routes.

2. **Intermediate:** Convert the campus delivery example from Tutorial 03 into a VRPTW by removing delivery locations (make all nodes "pickups" with dummy deliveries at depot).

3. **Advanced:** Create the same problem as VRP, CVRP, and PDPTW. Compare solution quality and computation time. When does adding constraints actually help?

**Hints:**
- For Exercise 1: Set capacity=5, demands=[3,3,3,3]
- For Exercise 2: Use the create_vrptw() function
- For Exercise 3: Use time.time() to measure solve time

In [ ]:
# Your solutions here


## 8. Summary

**What you learned:**
- ✅ Understand VRP problem hierarchy (VRP → CVRP → VRPTW → PDP → PDPTW)
- ✅ Model each variant using PDPTWInstance
- ✅ Convert between problem types
- ✅ Choose appropriate variant for your scenario

**Key takeaways:**
1. All variants can be modeled using PDPTWInstance with appropriate constraints
2. Simpler problems (VRP, CVRP) use "dummy deliveries" at depot
3. More constraints → harder problem, but often more realistic
4. Start simple, add complexity only when needed

**Next steps:**
- Try **Tutorial 06: Custom Algorithms** to implement solvers for specific variants
- Try **Tutorial 02: Real-World Maps** for realistic distance matrices
- Try **Tutorial 07: Data Generation** to create benchmark instances